In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KafkaSparkStreaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

transactions_raw_df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "darooghe.transactions") \
    .option("startingOffsets", "latest") \
    .load()

To implement real-time processing first we need to make an schema for the entering data. This can be made by the StructType type in pyspark library. 

In [10]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("location", StringType()),
    StructField("timestamp", TimestampType()),
    StructField("merchant_category", StringType()), 
    StructField("commission_type", StringType()), 
    StructField("commission_amount", DoubleType()), 
    StructField("merchant_id", StringType())
])

transactions_df = transactions_raw_df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")

Now we need to add a time-based window to the data frame and also make the desired insights from data. To do this we change the main dataframe.

we'll have two topics for the insights of the time sliding data; the amount topic and the merchant topic.

In [4]:
from pyspark.sql.functions import window, col, to_timestamp, count

# Group by 1-minute window with sliding every 20 seconds
transactions_df = transactions_df.withColumn("timestamp", to_timestamp(col("timestamp")))

amount = transactions_df.withWatermark("timestamp", "1 minutes") \
    .groupBy(
        window(col("timestamp"), "1 minutes", "20 seconds")
    ).agg(
        {"amount": "avg", "transaction_id": "count"}
    )
    
amount = amount.selectExpr(
    "CAST(window.start AS STRING) as key",
    "to_json(named_struct('start', window.start, 'end', window.end, 'avg_amount', `avg(amount)`, 'count_transactions', `count(transaction_id)`)) as value"
)

merchant = transactions_df.withWatermark("timestamp", "1 minute") \
    .groupBy(
        window(col("timestamp"), "1 minutes", "20 seconds"),
        col("merchant_category")
    ).agg(
        count("*").alias("transaction_count")
    )

merchant = merchant.selectExpr(
    "CAST(window.start AS STRING) as key",
    """
    to_json(named_struct(
        'window_start', window.start,
        'window_end', window.end,
        'merchant_category', merchant_category,
        'transaction_count', transaction_count
    )) as value
    """
)

In [ ]:
amount_query = amount.writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "amount_insights") \
    .option("checkpointLocation", "/tmp/checkpoints/amount_insights") \
    .option("startingOffsets", "latest") \
    .trigger(processingTime="10 seconds") \
    .start()

merchant_query = merchant.writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "merchant_insights") \
    .option("checkpointLocation", "/tmp/checkpoints/merchant_insights") \
    .option("startingOffsets", "latest") \
    .trigger(processingTime="10 seconds") \
    .start()
    
merchant_query.awaitTermination()

In [ ]:
merchant_query.stop()

## Fraud Detection System

In this part we'll load the transactions again and process them for fraud detection. 

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_timestamp, window,
    count, avg, struct, to_json, expr, udf
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import math

spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

schema = StructType([
    StructField("transaction_id",   StringType()),
    StructField("timestamp",        StringType()),
    StructField("customer_id",      StringType()),
    StructField("merchant_category",StringType()),
    StructField("amount",           DoubleType()),
    StructField("location",         StringType())
])
raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers","localhost:9092") \
    .option("subscribe","darooghe.transactions") \
    .option("startingOffsets","latest") \
    .load()

transactions_df = raw \
  .selectExpr("CAST(value AS STRING) as json") \
  .select(from_json("json", schema).alias("data")) \
  .select("data.*") \
  .withColumn("timestamp", to_timestamp(col("timestamp"))) \
  .withColumn("lat", expr("cast(get_json_object(location,'$.lat') as double)")) \
  .withColumn("lng", expr("cast(get_json_object(location,'$.lng') as double)")) \
  .withWatermark("timestamp","5 minutes")

25/04/28 19:22:48 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Velocity Check

In [6]:
velocityAlerts = transactions_df \
    .groupBy(
        window("timestamp","2 minutes","30 seconds"),
        col("customer_id")
    ).count() \
    .filter(col("count") > 5) \
    .selectExpr(
      "customer_id as key",
      """
        to_json(named_struct(
            'customer_id', customer_id,
            'transaction_count', count,
            'window_start', window.start,
            'window_end', window.end,
            'fraud_type', 'Velocity Check'
        )) as value
        """
    )

## Geographical Impossibility

In [6]:
from pyspark.sql.functions import udf, col, expr, to_json, struct
from pyspark.sql.types import DoubleType
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # km
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    Δφ = math.radians(lat2 - lat1)
    Δλ = math.radians(lon2 - lon1)
    a = math.sin(Δφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(Δλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

haversine_udf = udf(haversine, DoubleType())

geographical_alerts = transactions_df.alias("a").join(
    transactions_df.alias("b"),
    expr("""
        a.customer_id = b.customer_id
        AND a.timestamp < b.timestamp
        AND b.timestamp <= a.timestamp + interval 5 minutes
    """)
).filter(
    haversine_udf(col("a.lat"), col("a.lng"), col("b.lat"), col("b.lng")) > 50
).selectExpr(
    "CAST(a.customer_id AS STRING) as key",
    """
    to_json(named_struct(
        'transaction_id_1', a.transaction_id,
        'transaction_id_2', b.transaction_id,
        'lat1', a.lat,
        'lng1', a.lng,
        'lat2', b.lat,
        'lng2', b.lng,
        'timestamp1', a.timestamp,
        'timestamp2', b.timestamp,
        'fraud_type', 'Geographical Impossibility'
    )) as value
    """
)


## Amount Anomaly

In [7]:
from pyspark.sql.functions import avg

historical_df = spark.read.json("message.json")

customer_avg_df = historical_df.groupBy("customer_id").agg(
    avg("amount").alias("avg_amount")
)
amountAlerts = transactions_df.join(
    customer_avg_df, on="customer_id", how="left"
).filter(
    col("amount") > 10 * col("avg_amount")
).selectExpr(
    "CAST(customer_id AS STRING) as key",
    """
    to_json(named_struct(
        'transaction_id', transaction_id,
        'customer_id', customer_id,
        'amount', amount,
        'avg_amount', avg_amount,
        'timestamp', timestamp,
        'fraud_type', 'Amount Anomaly'
    )) as value
    """
)

## Uniting All The Alerts and Sending To Kafka

In [ ]:
from pyspark.sql import DataFrame

fraudAlerts = velocityAlerts.union(geographical_alerts).union(amountAlerts)

fraud_query = fraudAlerts.writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers","localhost:9092") \
    .option("topic","darooghe.fraud_alerts") \
    .option("checkpointLocation","/tmp/checkpoints/fraud_alerts") \
    .trigger(processingTime="30 seconds") \
    .start()

fraud_query.awaitTermination()

In [30]:
fraud_query.stop()

## Real-Time Commission Analytics

### Total Commission By Type Per Minute

In [11]:
from pyspark.sql.functions import sum

commission_by_type = transactions_df \
    .withWatermark("timestamp", "2 minutes") \
    .groupBy(
        window(col("timestamp"), "1 minute", "30 seconds"),  # sliding باشه بهتره
        col("commission_type")
    ).agg(
        sum("commission_amount").alias("total_commission")
    )

commission_by_type = commission_by_type.selectExpr(
    "CAST(commission_type AS STRING) as key",
    """
    to_json(named_struct(
        'window_start', window.start,
        'window_end', window.end,
        'commission_type', commission_type,
        'total_commission', total_commission
    )) as value
    """
)

### Commission Ratio By Merchant Category

In [12]:
commission_ratio_by_category = transactions_df \
    .withWatermark("timestamp", "2 minutes") \
    .groupBy(
        window(col("timestamp"), "1 minute", "30 seconds"),
        col("merchant_category")
    ).agg(
        (sum("commission_amount") / sum("amount")).alias("commission_ratio")
    )

commission_ratio_by_category = commission_ratio_by_category.selectExpr(
    "CAST(merchant_category AS STRING) as key",
    """
    to_json(named_struct(
        'window_start', window.start,
        'window_end', window.end,
        'merchant_category', merchant_category,
        'commission_ratio', commission_ratio
    )) as value
    """
)

### Highest Commission-generating Merchants

In [ ]:
from pyspark.sql.functions import window, sum, col

merchant_commissions = transactions_df \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window(col("timestamp"), "5 minutes", "1 minute"),
        col("merchant_id")
    ).agg(
        sum("commission_amount").alias("total_commission")
    )

top_merchants = merchant_commissions \
    .orderBy(col("total_commission").desc()) \
    .limit(5)

top_merchants = top_merchants.selectExpr(
    "CAST(merchant_id AS STRING) as key",
    """
    to_json(named_struct(
        'window_start', window.start,
        'window_end', window.end,
        'merchant_id', merchant_id,
        'total_commission', total_commission
    )) as value
    """
)

### Writing Commision Insights On Kafka

In [ ]:
# A
commission_by_type_query = commission_by_type.writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "commission_by_type") \
    .option("checkpointLocation", "/tmp/checkpoints/commission_by_type") \
    .trigger(processingTime="30 seconds") \
    .start()

# B
commission_ratio_query = commission_ratio_by_category.writeStream \
    .outputMode("append") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "commission_ratio_by_category") \
    .option("checkpointLocation", "/tmp/checkpoints/commission_ratio_by_category") \
    .trigger(processingTime="30 seconds") \
    .start()

# C
top_merchants_query = top_merchants.writeStream \
    .outputMode("complete") \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "top_merchants") \
    .option("checkpointLocation", "/tmp/checkpoints/top_merchants") \
    .trigger(processingTime="30 seconds") \
    .start()

commission_ratio_query.awaitTermination()

25/04/28 19:44:22 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/04/28 19:44:23 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/04/28 19:44:24 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/04/28 19:44:25 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.


In [16]:
commission_ratio_query.stop()

25/04/28 19:44:30 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/04/28 19:44:31 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/04/28 19:44:32 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/04/28 19:44:33 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/04/28 19:44:34 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/04/28 19:44:35 WARN NetworkClient: [Producer clientId=producer-1] Connection to node 1 (localhost/127.0.0.1:9092) could not be establishe